In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
df_test = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_test_with_ESM1b_ts.pkl")
)
df_test = df_test.loc[df_test["ESM1b"] != ""]
df_test.reset_index(inplace=True, drop=True)

df_train = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "splits", "df_train_with_ESM1b_ts.pkl")
)
df_train = df_train.loc[df_train["ESM1b"] != ""]
df_train.reset_index(inplace=True, drop=True)

/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)
/home/hanxd/miniconda3/envs/esp/lib/python3.8/site-packages/pandas/core/ops/array_ops.py:82: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  result = libops.scalar_compare(x.ravel(), y, op)


In [ ]:
p450plant0 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant0.pkl")
)
p450plant1 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant1.pkl")
)
p450plant2 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant2.pkl")
)
p450plant3 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant3.pkl")
)
p450plant4 = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "p450plant4.pkl")
)

p450plant = pd.concat(
    [p450plant0, p450plant1, p450plant2, p450plant3, p450plant4], ignore_index=True
)

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]

In [ ]:
train_X, train_y = create_input_and_output_data(df=df_train)
test_X, test_y = create_input_and_output_data(df=df_test)

test_new_X, test_new_y = create_input_and_output_data(df=p450plant)


from sklearn.model_selection import train_test_split

# same split as in Mou et al paper:


train_X = np.concatenate([train_X, test_new_X])
train_y = np.concatenate([train_y, test_new_y])

In [ ]:
param = {
    "learning_rate": 0.60553117247348733,
    "max_delta_step": 1.7726044219753656,
    "max_depth": 10,
    "min_child_weight": 1.3845040588450772,
    "num_rounds": 342.68325188584106,
    "reg_alpha": 0.531395259755843,
    "reg_lambda": 3.744980563764689,
    "weight": 0.26187490421514203,
}

num_round = param["num_rounds"]
# param["tree_method"] = "gpu_hist"
# param["sampling_method"] = "gradient_based"
param["objective"] = "binary:logistic"
param["eval_metric"] = ["error", "logloss"]

weights1 = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in df_train["Binding"]]
)
weights2 = np.array(
    [param["weight"] if binding == 0 else 1.0 for binding in test_new_y]
)

weights = np.concatenate([weights1, weights2])


del param["num_rounds"]
del param["weight"]

dtrain = xgb.DMatrix(
    np.array(train_X),
    weight=weights,
    label=np.array(train_y),
    feature_names=feature_names,
)
dtest = xgb.DMatrix(
    np.array(test_X), label=np.array(test_y), feature_names=feature_names
)

evallist = [(dtrain, "train")]

bst = xgb.train(param, dtrain, int(num_round), evallist, verbose_eval=10)
y_test_pred = np.round(bst.predict(dtest))
acc_test = np.mean(y_test_pred == np.array(test_y))
roc_auc = roc_auc_score(np.array(test_y), bst.predict(dtest))

print("Accuracy on test set: %s, ROC-AUC score for test set: %s" % (acc_test, roc_auc))

[0]	train-error:0.28614	train-logloss:0.58752
[10]	train-error:0.11488	train-logloss:0.36136
[20]	train-error:0.06039	train-logloss:0.25599
[30]	train-error:0.03669	train-logloss:0.19834
[40]	train-error:0.02530	train-logloss:0.15987
[50]	train-error:0.01687	train-logloss:0.12874
[60]	train-error:0.01193	train-logloss:0.10810
[70]	train-error:0.00888	train-logloss:0.09180
[80]	train-error:0.00659	train-logloss:0.07931
[90]	train-error:0.00507	train-logloss:0.06885
[100]	train-error:0.00399	train-logloss:0.05904
[110]	train-error:0.00337	train-logloss:0.05275
[120]	train-error:0.00284	train-logloss:0.04679
[130]	train-error:0.00240	train-logloss:0.04262
[140]	train-error:0.00217	train-logloss:0.03901
[150]	train-error:0.00201	train-logloss:0.03573
[160]	train-error:0.00179	train-logloss:0.03311
[170]	train-error:0.00163	train-logloss:0.03090
[180]	train-error:0.00157	train-logloss:0.02881
[190]	train-error:0.00141	train-logloss:0.02723
[200]	train-error:0.00133	train-logloss:0.02577
[21

In [ ]:
pickle.dump(bst, open(join(our_data + "p450normalmodel.dat"), "wb"))